# Context-Aware Chatbot Using LangChain and Retrieval-Augmented Generation (RAG)

## Problem Statement

Traditional chatbots often struggle to maintain conversational context and provide accurate responses based on external knowledge sources. They are typically limited to information contained within their pre-trained models and may fail to answer domain-specific questions accurately. A more intelligent solution is needed that can remember previous interactions and retrieve relevant information from external documents in real time.

## Objective

The objective of this project is to develop a context-aware conversational chatbot using LangChain and Retrieval-Augmented Generation (RAG). The chatbot should maintain conversation history, retrieve relevant information from a vectorized knowledge base, and generate accurate responses based on both user context and external documents. The final system will be deployed using Streamlit to provide an interactive user experience.


In [ ]:
# ==========================================
# INSTALL REQUIRED LIBRARIES
# ==========================================
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install langchain-text-splitters
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install accelerate

In [19]:

# Imports
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

# ==========================================================
# KNOWLEDGE BASE
# ==========================================================

knowledge_base = """

Artificial Intelligence (AI) is the simulation of human intelligence by machines.

Machine Learning (ML) is a subset of AI that allows systems to learn from data.

Deep Learning uses neural networks with many layers to learn complex patterns.

Natural Language Processing (NLP) enables computers to understand and generate human language.

LangChain is an open-source framework for building applications powered by Large Language Models (LLMs).

Retrieval-Augmented Generation (RAG) combines retrieval systems with language models to improve answer quality.

FAISS is a vector database developed by Meta for semantic search and similarity retrieval.

Hugging Face provides open-source transformer models, datasets, and NLP tools.

Context-aware chatbots remember previous interactions and use them to generate better responses.

"""

# ==========================================================
# CREATE DOCUMENTS
# ==========================================================

documents = [Document(page_content=knowledge_base)]

# ==========================================================
# SPLIT INTO CHUNKS
# ==========================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

# ==========================================================
# EMBEDDINGS
# ==========================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ==========================================================
# VECTOR DATABASE
# ==========================================================

vector_db = FAISS.from_documents(
    docs,
    embedding_model
)

# ==========================================================
# HUGGING FACE MODEL
# ==========================================================

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    do_sample=False
)

# ==========================================================
# MEMORY
# ==========================================================

chat_history = []

# ==========================================================
# RETRIEVAL FUNCTION
# ==========================================================

def retrieve_context(query):

    results = vector_db.similarity_search(
        query,
        k=2
    )

    return "\n".join(
        [doc.page_content for doc in results]
    )

# ==========================================================
# CHATBOT FUNCTION
# ==========================================================

def chatbot(query):

    global chat_history

    context = retrieve_context(query)

    history = "\n".join(chat_history[-4:])

    prompt = f"""
Answer briefly using ONLY the context below.

Context:
{context}

Conversation History:
{history}

Question:
{query}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=25,
        return_full_text=False,
        do_sample=False
    )

    answer = result[0]["generated_text"].strip()
    # Clean extra generated text
    for stop_word in ["Question:", "Answer:", "User:", "Assistant:"]:
          if stop_word in answer:
            answer = answer.split(stop_word)[0]

    answer = answer.strip()
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: {answer}")

    return answer

# ==========================================================
# INTERACTIVE CHAT
# ==========================================================

print("Chatbot Ready!")
print("Type 'exit' to quit.\n")

while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Chat Ended")
        break

    print("\nBot:", chatbot(question))
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Chatbot Ready!
Type 'exit' to quit.

You: What is ML


[transformers] Both `max_new_tokens` (=25) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Bot: Machine Learning (ML) is a subset of AI that allows systems to learn from data.

You: exit
Chat Ended


In [20]:
## Streamlit deployment

%%writefile app.py
import streamlit as st
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import pipeline

# ==========================================================
# PAGE CONFIG
# ==========================================================

st.set_page_config(
    page_title="Context-Aware RAG Chatbot",
    page_icon="🤖",
    layout="wide"
)

st.title("🤖 Context-Aware RAG Chatbot")
st.write("Task 4 - LangChain + RAG + Memory")

# ==========================================================
# KNOWLEDGE BASE
# ==========================================================

knowledge_base = """
Artificial Intelligence (AI) is the simulation of human intelligence by machines.

Machine Learning (ML) is a subset of AI that allows systems to learn from data.

Deep Learning uses neural networks with many layers to learn complex patterns.

Natural Language Processing (NLP) enables computers to understand and generate human language.

LangChain is an open-source framework for building applications powered by Large Language Models (LLMs).

Retrieval-Augmented Generation (RAG) combines retrieval systems with language models to improve answer quality.

FAISS is a vector database developed by Meta for semantic search and similarity retrieval.

Hugging Face provides open-source transformer models, datasets, and NLP tools.

Context-aware chatbots remember previous interactions and use them to generate better responses.
"""

# ==========================================================
# LOAD MODELS ONLY ONCE
# ==========================================================

@st.cache_resource
def load_resources():

    docs = [Document(page_content=knowledge_base)]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=300,
        chunk_overlap=50
    )

    split_docs = splitter.split_documents(docs)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vector_db = FAISS.from_documents(
        split_docs,
        embeddings
    )

    generator = pipeline(
        "text-generation",
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        do_sample=False
    )

    return vector_db, generator

vector_db, generator = load_resources()

# ==========================================================
# MEMORY
# ==========================================================

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# ==========================================================
# RETRIEVAL
# ==========================================================

def retrieve_context(query):

    results = vector_db.similarity_search(query, k=2)

    return "\n".join(
        [doc.page_content for doc in results]
    )

# ==========================================================
# CHATBOT
# ==========================================================

def chatbot(query):

    context = retrieve_context(query)

    history = "\n".join(
        st.session_state.chat_history[-4:]
    )

    prompt = f"""
Answer briefly using ONLY the context below.

Context:
{context}

Conversation History:
{history}

Question:
{query}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=50,
        return_full_text=False
    )

    answer = result[0]["generated_text"].strip()

    st.session_state.chat_history.append(
        f"User: {query}"
    )

    st.session_state.chat_history.append(
        f"Assistant: {answer}"
    )

    return answer

# ==========================================================
# CHAT UI
# ==========================================================

user_query = st.chat_input("Ask something...")

if user_query:

    answer = chatbot(user_query)

    st.chat_message("user").write(user_query)
    st.chat_message("assistant").write(answer)

# ==========================================================
# DISPLAY HISTORY
# ==========================================================

with st.expander("Conversation History"):
    for item in st.session_state.chat_history:
        st.write(item)

Overwriting app.py


In [6]:
!pip install streamlit pyngrok langchain langchain-community langchain-core langchain-text-splitters sentence-transformers faiss-cpu transformers accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 49.4 MB/s eta 0:00:00


In [21]:
!streamlit run app.py >/content/logs.txt &

2026-06-21 16:37:30.958 Uvicorn server started on 0.0.0.0:8501


In [22]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [24]:
!streamlit run app.py \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.address 0.0.0.0 \
  --server.port 8501 &



2026-06-21 16:38:05.082 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.73.182.39:8501

/content/app.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/content/app.py:61: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%

# Final Summary and Insights

## Project Summary

A context-aware chatbot was successfully developed using LangChain and Retrieval-Augmented Generation (RAG). The system combines Large Language Models (LLMs), vector-based document retrieval, and conversational memory to provide accurate and contextually relevant responses.

## Methodology

- Loaded and processed external knowledge documents.
- Converted documents into vector embeddings.
- Stored embeddings in a vector database for efficient retrieval.
- Implemented Retrieval-Augmented Generation (RAG) using LangChain.
- Added conversation memory to maintain context across interactions.
- Built an interactive user interface using Streamlit.

## Key Results

- Successfully retrieves relevant information from the knowledge base.
- Maintains conversation history for context-aware responses.
- Produces more accurate and grounded answers compared to a standard chatbot.
- Provides an interactive and user-friendly conversational experience.

## Insights

- RAG significantly improves response quality by grounding answers in external documents.
- Conversation memory enhances user experience by enabling multi-turn interactions.
- Vector databases enable efficient semantic search and retrieval.
- LangChain simplifies the integration of LLMs, retrieval systems, and memory components.

## Challenges Encountered

- Selecting appropriate chunk sizes for document splitting.
- Balancing retrieval accuracy and response generation speed.
- Managing context length limitations of large language models.

## Future Improvements

- Support multiple document formats (PDF, DOCX, TXT).
- Implement hybrid search combining keyword and semantic retrieval.
- Add user authentication and chat history storage.
- Integrate more advanced vector databases such as FAISS, Chroma, or Pinecone.
- Deploy the chatbot on cloud platforms for public access.

## Conclusion

The project demonstrates how Retrieval-Augmented Generation (RAG) and conversational memory can be combined to create an intelligent, context-aware chatbot capable of delivering accurate, relevant, and personalized responses based on both user interactions and external knowledge sources.